# steering-qwen: CUDA check on Colab

This notebook checks that the repo works on an NVIDIA GPU. It uses the exact locked environment (`uv sync --locked`), not Colab's preinstalled packages.

1. **Runtime → Change runtime type → T4 GPU** (free). An L4 or A100 is a closer match if you have one.
2. **Runtime → Run all.** This takes about 10 minutes: roughly 4 GB of packages, then the 9 GB model.
3. Read the **Summary** at the bottom.

What it runs:

| Stage | What it proves |
|---|---|
| GPU + install | Locked torch, bitsandbytes and flash-linear-attention install and see the GPU |
| `pytest` | The tiny-Qwen3.5 tests run on `cuda` too, in bf16 and in bitsandbytes 4-bit |
| Real model, NF4 | Qwen3.5-4B in 4-bit on CUDA gives the notebook's calibration numbers, steering works, and clearing restores the baseline |
| Real model, bf16 | The same checks for `--no-quantize` |
| CLI | `uv run steer` works end to end |

If the GPU or install stage fails, the later stages are skipped rather than quietly running on CPU.

**T4 caveat:** the T4 (`sm_75`) has no native bf16, and Triton's bf16 support on it is limited. If only the flash-linear-attention / bf16 stages fail on a T4, rerun on an L4 or A100 before blaming the code.

In [ ]:
# Which code to test (point these at a fork and branch to test changes before merging)
REPO_URL = "https://github.com/ramayer/steering-qwen.git"
BRANCH = "main"
DIR = "/content/steering-qwen"

In [ ]:
import os, subprocess, time

os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", PYTHONUNBUFFERED="1", UV_LINK_MODE="copy")
RESULTS = []    # (stage, PASS/FAIL/SKIP, seconds)
BLOCKED = None  # a failed gating stage; everything after it is skipped

def run(name, cmd, cwd=None, gate=False):
    """Run a shell command, stream its output, and record the result for the summary."""
    global BLOCKED
    if BLOCKED:
        RESULTS.append((name, "SKIP", 0.0))
        print(f"--> {name}: SKIP (needs '{BLOCKED}')")
        return False
    print(f"$ {cmd}", flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    status = "PASS" if rc == 0 else "FAIL"
    RESULTS.append((name, status, time.time() - t0))
    print(f"\n--> {name}: {status}{'' if rc == 0 else f' (exit {rc})'} in {time.time() - t0:.0f}s", flush=True)
    if rc != 0 and gate:
        BLOCKED = name
    return rc == 0

if not run("GPU present", "nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv", gate=True):
    print("\nNo GPU. Use Runtime -> Change runtime type -> T4 GPU, then Run all again.")

In [ ]:
run("install",
    f"pip install -q -U uv && rm -rf {DIR} && git clone -q --depth 1 -b {BRANCH} {REPO_URL} {DIR} "
    f"&& cd {DIR} && git log --oneline -1 && uv sync --locked",
    gate=True)
run("versions + CUDA visible", """uv run python -c "
import torch, bitsandbytes, transformers, triton, fla
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| bitsandbytes', bitsandbytes.__version__,
      '| transformers', transformers.__version__, '| triton', triton.__version__, '| fla', getattr(fla, '__version__', '?'))
assert torch.cuda.is_available(), 'torch cannot see the GPU'
cap = torch.cuda.get_device_capability()
print(torch.cuda.get_device_name(), f'sm_{cap[0]}{cap[1]}', '| bf16 native:', cap >= (8, 0))
" """, cwd=DIR, gate=True)

In [ ]:
run("pytest (tiny Qwen3.5 on cpu + cuda)", "REQUIRE_CUDA=1 uv run pytest -v -rs", cwd=DIR)

In [ ]:
run("real model, 4-bit NF4", "uv run scripts/check_against_notebook.py --device cuda --quantize", cwd=DIR)

In [ ]:
run("real model, bf16", "uv run scripts/check_against_notebook.py --device cuda --no-quantize", cwd=DIR)

In [ ]:
run("CLI", 'uv run steer --device cuda --baseline -n 60 -s 14:8949:7.5 "What do you think about humans?"', cwd=DIR)

## Summary

In [ ]:
gpu = subprocess.run("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip() or "no GPU"
commit = subprocess.run(f"git -C {DIR} log --oneline -1", shell=True, capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu}\nCommit: {commit or '?'}\n")
for name, status, secs in RESULTS:
    print(f"{status:4}  {name:40} {secs:6.0f}s")
ok = RESULTS and all(s == "PASS" for _, s, _ in RESULTS)
print("\nALL PASSED" if ok else "\nNOT ALL PASSED: scroll up to the first FAIL")